In [7]:
import ot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import math
import scipy
from scipy.spatial import cKDTree
from scipy import integrate
from scipy import interpolate
from scipy.spatial import Delaunay
from scipy.spatial import Voronoi
from itertools import product
import torch
torch.manual_seed(100000)
import torch.nn as nn
import torch.optim as optim



In [8]:
#settings
start = np.array((0., 1., 1.05))
dt = 0.01
subset_size = 20#50 #cell number
slope = 0.5#parameter in weight computation
traj_length = int(1e7)#length of trajectory
noise_level = 0.5
sample_size = int(500)
epsilon = 1e-4
sampling_time = 1
simulation_t =int(2e4)
steps = 5
N_iters = 10000
costs1W, costs2W, costs1L, costs2L = [],[],[],[]

In [ ]:

# Define the Lorenz-63 System
def Lorenz(x):
    x = x.T
    dx = 10 * (x[1] - x[0])
    dy = x[0] * (28 - x[2]) - x[1]
    dz = x[0] * x[1] - (8/3) * x[2]
    f = np.array((dx, dy, dz)).T
    return f

 #trajectory simulation (Euler's method)
def traj(length, starts = start, model = Lorenz):
    x = starts
    traj = np.zeros((length,len(starts)))
    traj[0] = np.array(x)
    for timestep in range(length-1):

        x = x + dt * model(x)

        traj[timestep+1] = np.array(x)

    return traj

def invariant_measure(matrix):
    N = len(matrix)
    rhs = (-1) * (epsilon/N) * torch.ones(N)
    rho = torch.linalg.solve(((1 - epsilon) * matrix - torch.eye(N)),rhs)
    return rho


relu = nn.ReLU()
def decay(x):
    return relu(1-slope*x)
def w(xs):
    dists =  torch.cdist(xs, Voronoi_centers, p =2)
    pre_w = decay(dists)
    return  pre_w / torch.sum(pre_w, dim=1,keepdim = True)


def Ulam(points,Tpoints):
    mat = torch.zeros((subset_size, subset_size))
    #before normalization
    with torch.no_grad():
          randpts_idxs = torch.tensor(tree.query(points.detach().numpy())[1], dtype=torch.int)
    weights = w(Tpoints)
    mat.index_add_(0, randpts_idxs, weights)
    mat = (mat.T)/mat.sum(dim = 1)
    return mat

class W2Loss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, U_net):
        # Save input for backward
        U_net_np = U_net.detach().numpy()

        # Normalize inputs to sum to 1
        a = U_true_np.flatten()
        b = U_net_np.flatten()
        a = a / np.sum(a)
        b = b / np.sum(b)

        n = subset_size
        X, Y = np.meshgrid(np.arange(n), np.arange(n))
        coords = np.stack([X.ravel(), Y.ravel()], axis=1)
        costM = ot.dist(coords, coords, metric='sqeuclidean') 
        _, log = ot.emd(a, b, costM, log=True)

        cost, grad = log['cost'],log['v']
        loss = cost
        grad_tensor = torch.tensor(grad, dtype=U_net.dtype).reshape_as(U_net)
        ctx.save_for_backward(grad_tensor)
        return torch.tensor(loss, dtype=U_net.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        grad_tensor, = ctx.saved_tensors  # Unpack gradient
        return grad_tensor.reshape(subset_size,subset_size)
trajectory_clean = traj(traj_length)

trajectory = trajectory_clean + np.random.normal(0,noise_level,((traj_length,3)))#loooooooong trajectory
trajectory = trajectory[int(1e4):]

In [ ]:
    rand_idxs = np.random.choice(len(trajectory), size=sample_size, replace=False)
    observed = trajectory[rand_idxs]
    randpts = torch.tensor(trajectory[rand_idxs],dtype = torch.float)
    Trandpts = torch.tensor(trajectory[rand_idxs+steps],dtype = torch.float)

#normalization
    M_scale = torch.max(torch.abs(randpts))
    randpts /= M_scale
    Trandpts /= M_scale

#Voronoi cell center
    Voronoi_centers = MiniBatchKMeans(n_clusters=subset_size).fit(randpts).cluster_centers_

    tree = cKDTree(Voronoi_centers)
    Voronoi_centers = torch.tensor(Voronoi_centers,dtype = torch.float)
    U_true = Ulam(randpts,Trandpts)
    U_true_np = U_true.detach().cpu().numpy()

In [ ]:
for iteration in range(10):
    net1 = nn.Sequential(
    nn.Linear(3, 100),
    nn.Tanh(),
    nn.Linear(100, 100),
    nn.Tanh(),
    nn.Linear(100, 100),
    nn.Tanh(),
    nn.Linear(100, 3)
)

    optimizer1 = optim.Adam(net1.parameters(), lr=1e-3)
    net1.train()
    loss1 = []

# Store the first loss
    net1_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field = net1(net1_randpts)
        net1_randpts = net1_randpts + dt * V_field  # out-of-place update
    U_net = Ulam(randpts, net1_randpts)
    initial_L1 = W2Loss.apply(U_net)

    for i in range(N_iters):
        optimizer1.zero_grad()
        net1_randpts = randpts

        for _ in range(steps):
            V_field = net1(net1_randpts)
            net1_randpts = net1_randpts + dt * V_field

        U_net = Ulam(randpts, net1_randpts)
    
    # Compute loss
        L1 = W2Loss.apply(U_net)
    

        L1.backward()
        optimizer1.step()
        loss1.append(L1.item())

    # Logging
        with torch.no_grad():
            if i % 500 == 0:
                if L1.item() < 0.02 * initial_L1:
                    break
    x1 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
    vals1 = [x1.detach().numpy().flatten()]
    for _ in range(simulation_t-1):

        V_field = net1(x1)
        x1 = x1+dt* V_field
        vals1.append(x1.detach().numpy().flatten())
    vals1 = np.array(vals1)
    net2 = nn.Sequential(
            nn.Linear(3, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 3))


    optimizer2 = optim.Adam(net2.parameters(),lr = 1e-3)
    net2.train()
    loss2 = []

# Store the first loss
    net2_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field2 = net2(net2_randpts)
        net2_randpts = net2_randpts + dt * V_field2  # out-of-place update
    initial_L2 = torch.mean((net2_randpts - Trandpts) ** 2)
    for i in range(N_iters):
    # Zero gradients for each optimizer
        optimizer2.zero_grad()
        net2_randpts = randpts
        for _ in range(steps):
            V_field2 = net2(net2_randpts)
            net2_randpts = net2_randpts+dt* V_field2

        L2 = torch.mean((net2_randpts - Trandpts) ** 2)
        L2.backward()
        optimizer2.step()
        loss2.append(L2.item())
    # Logging
        if i % 500 == 0:
            if L2.item() < 0.02 * initial_L2:
                break
    x2 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
    vals2 = [x2.detach().numpy().flatten()]
    for _ in range(simulation_t-1):

        V_field2 = net2(x2)
        x2 = x2 + V_field2*dt
        vals2.append(x2.detach().numpy().flatten())
    vals2 = np.array(vals2)
    GT = traj(int(simulation_t), M_scale* np.array(randpts[0]))
    
    subGT, subvals1, subvals2 = GT, np.array(M_scale * vals1), np.array(M_scale * vals2)
    a, b = np.ones((simulation_t,)) / simulation_t, np.ones((simulation_t,)) / simulation_t
    M1, M2 = ot.dist(subGT, subvals1),ot.dist(subGT, subvals2)
    _,G01 = ot.emd(a, b, M1,log = 'true')
    _,G02 = ot.emd(a, b, M2,log = 'true')
    cost1W,cost2W = (G01['cost'])**(1/2),(G02['cost'])**(1/2)
    GTpts = torch.tensor((GT/M_scale)[:-steps],dtype = torch.float)
    GTTpts = GT[steps:]

    net1pts,net2pts = GTpts, GTpts
    for i in range(steps):
        Vf1,Vf2 = net1(net1pts),net2(net2pts)
        net1pts,net2pts = net1pts+dt*Vf1,net2pts+dt*Vf2
    cost1L = (np.mean((np.array(M_scale)*net1pts.detach().numpy()-GTTpts)**2))**(1/2)
    cost2L = (np.mean((np.array(M_scale)*net2pts.detach().numpy()-GTTpts)**2))**(1/2)
    print("Experiment ", iteration,": W2(ours) ",cost1W, ", W2(pointwise) ",cost2W)
    print("Experiment ", iteration,": L2(ours) ",cost1L, ", L2(pointwise) ",cost2L)
    costs1W.append(cost1W)
    costs2W.append(cost2W)
    costs1L.append(cost1L)
    costs2L.append(cost2L)

In [ ]:
costs1W,costs2W,costs1L,costs2L

In [ ]:
np.mean(costs1W),np.std(costs1W)

In [ ]:
np.mean(costs2W),np.std(costs2W)

In [ ]:
np.mean(costs1L),np.std(costs1L)

In [ ]:
np.mean(costs2L),np.std(costs2L)